In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator, MACCSkeys
from sklearn.feature_selection import VarianceThreshold
from tqdm.auto import tqdm
import urllib.request
import tarfile
import os
# --- 1. DATA LOADING AND CLEANING ---
url          = "https://sid.erda.dk/share_redirect/c7LF5NaYvH"
archive_name = "data.tar.xz"
path_elec    = "data/QMdata4ML/df_elec.csv.gz"

if not os.path.exists(path_elec):
    if not os.path.exists(archive_name):
        print("Downloading archive (~1.1 GB)...")
        urllib.request.urlretrieve(url, archive_name)
        print("Download complete.")
    print("Extracting archive...")
    with tarfile.open(archive_name, "r:xz") as tar:
        tar.extractall()
    print("Extraction complete.")

df_elec = pd.read_csv(path_elec)
print(f"Loaded {len(df_elec)} rows.")
try:
    display(df_elec.head())
except NameError:
    print(df_elec.head())

df_final = df_elec.drop(columns=['elec_GCS_3_cm5', 'Set', 'elec_names', 'Unnamed: 0'], errors='ignore').copy().reset_index(drop=True)

print(f"✅ Dataset loaded: {len(df_final)} sites | Unique molecules: {df_final['smiles'].nunique()}")

# --- 2. FEATURE GENERATION (MORGAN & MACCS) ---
print("\n🧪 Calculating Morgan Fingerprints (2048 bits)...")
mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
X_morgan_raw = np.array([
    mfpgen.GetFingerprintAsNumPy(Chem.MolFromSmiles(s)) if Chem.MolFromSmiles(s) else np.zeros(2048, dtype=np.int8)
    for s in tqdm(df_final['smiles'])
])

print("\n🧬 Calculating MACCS keys (167 bits)...")
X_maccs = np.array([
    list(MACCSkeys.GenMACCSKeys(Chem.MolFromSmiles(s))) if Chem.MolFromSmiles(s) else [0] * 167 
    for s in tqdm(df_final['smiles'])
], dtype=np.int8)

# --- 3. FILTERING ZERO-VARIANCE COLUMNS (Morgan only) ---
selector = VarianceThreshold(threshold=0.0)
X_morgan = selector.fit_transform(X_morgan_raw)
print(f"\n📉 Morgan filtering complete: {X_morgan_raw.shape[1]} -> {X_morgan.shape[1]} useful columns.")

# --- 4. PREPARING TARGETS AND GROUPS ---
y = df_final['MAA_values'].values
groups = df_final['smiles'].values

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

print("📦 Canonical Group Split by Molecule (80/10/10)...")

# --- 1. ISOLATING TEST SET (10%) ---
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=42)
train_val_idx, test_idx = next(gss_test.split(X_morgan, y, groups=groups))

# --- 2. ISOLATING VALIDATION SET (11.11% of the remaining 90% = 10% total) ---
groups_train_val = groups[train_val_idx]
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.1111, random_state=42)
train_idx, val_idx = next(gss_val.split(train_val_idx, y[train_val_idx], groups=groups_train_val))

# Remapping to global indices to slice our matrices cleanly
global_train_idx = train_val_idx[train_idx]
global_val_idx = train_val_idx[val_idx]
global_test_idx = test_idx

# --- 3. APPLYING SPLIT TO FEATURES ---
# For Morgan
X_train_morgan = X_morgan[global_train_idx]
X_val_morgan = X_morgan[global_val_idx]
X_test_morgan = X_morgan[global_test_idx]

# For MACCS
X_train_maccs = X_maccs[global_train_idx]
X_val_maccs = X_maccs[global_val_idx]
X_test_maccs = X_maccs[global_test_idx]

# For targets
y_train = y[global_train_idx]
y_val = y[global_val_idx]
y_test = y[global_test_idx]

# --- 4. SAFETY LEAKAGE CHECK ---
train_smiles = set(groups[global_train_idx])
val_smiles = set(groups[global_val_idx])
test_smiles = set(groups[global_test_idx])

print(f"• Train : {len(global_train_idx)} sites ({len(train_smiles)} molecules)")
print(f"• Val   : {len(global_val_idx)} sites ({len(val_smiles)} molecules)")
print(f"• Test  : {len(global_test_idx)} sites ({len(test_smiles)} molecules)")

assert train_smiles.isdisjoint(val_smiles) and train_smiles.isdisjoint(test_smiles), "❌ Molecular leakage detected!"
print("\n✅ SECURITY: No SMILES overlap. The sets are perfectly isolated.")

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

print("📦 Canonical Group Split by Molecule (80/10/10)...")

# --- 1. ISOLATING TEST SET (10%) ---
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=42)
train_val_idx, test_idx = next(gss_test.split(X_morgan, y, groups=groups))

# --- 2. ISOLATING VALIDATION SET (11.11% of the remaining 90% = 10% total) ---
groups_train_val = groups[train_val_idx]
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.1111, random_state=42)
train_idx, val_idx = next(gss_val.split(train_val_idx, y[train_val_idx], groups=groups_train_val))

# Remapping to global indices to slice our matrices cleanly
global_train_idx = train_val_idx[train_idx]
global_val_idx = train_val_idx[val_idx]
global_test_idx = test_idx

# --- 3. APPLYING SPLIT TO FEATURES ---
# For Morgan
X_train_morgan = X_morgan[global_train_idx]
X_val_morgan = X_morgan[global_val_idx]
X_test_morgan = X_morgan[global_test_idx]

# For MACCS
X_train_maccs = X_maccs[global_train_idx]
X_val_maccs = X_maccs[global_val_idx]
X_test_maccs = X_maccs[global_test_idx]

# For targets
y_train = y[global_train_idx]
y_val = y[global_val_idx]
y_test = y[global_test_idx]

# --- 4. SAFETY LEAKAGE CHECK ---
train_smiles = set(groups[global_train_idx])
val_smiles = set(groups[global_val_idx])
test_smiles = set(groups[global_test_idx])

print(f"• Train : {len(global_train_idx)} sites ({len(train_smiles)} molecules)")
print(f"• Val   : {len(global_val_idx)} sites ({len(val_smiles)} molecules)")
print(f"• Test  : {len(global_test_idx)} sites ({len(test_smiles)} molecules)")

assert train_smiles.isdisjoint(val_smiles) and train_smiles.isdisjoint(test_smiles), "❌ Molecular leakage detected!"
print("\n✅ SECURITY: No SMILES overlap. The sets are perfectly isolated.")

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# --- 1. MODEL DEFINITIONS ---
ranf_reg = RandomForestRegressor(
    n_estimators=100, max_depth=10, max_features='sqrt', 
    n_jobs=6, min_samples_leaf=5, random_state=42
)

xgb_reg = XGBRegressor(
    n_estimators=100, max_depth=6, learning_rate=0.1, 
    subsample=0.8, colsample_bytree=0.8, n_jobs=6, random_state=42
)

xgb_reg_GridSearchCV = XGBRegressor(
    n_estimators=300, max_depth=7, learning_rate=0.1, 
    subsample=0.8, colsample_bytree=0.8, n_jobs=6, random_state=42
)

xgb_reg_OPTUNA = XGBRegressor(
    n_estimators=1480, max_depth=9, learning_rate=0.060094, 
    subsample=0.647226, colsample_bytree=0.8, n_jobs=6, random_state=42
)

xgb_maccs = XGBRegressor(
    n_estimators=1000, max_depth=7, learning_rate=0.05, 
    tree_method='hist', n_jobs=6, random_state=42
)

# --- 2. TRAINING AND EVALUATION FUNCTION ---
all_results = []

def train_test_model(model, X_train_data, y_train_data, X_test_data, y_test_data, name="Model"):
    with tqdm(total=3, desc=f"⏳ {name}") as pbar:
        # Training
        model.fit(X_train_data, y_train_data)
        pbar.update(1)
        
        # Prediction
        y_pred_train = model.predict(X_train_data)
        y_pred_test = model.predict(X_test_data)
        pbar.update(1)
        
        # Metrics
        train_rmse = mean_squared_error(y_train_data, y_pred_train) ** 0.5
        test_rmse = mean_squared_error(y_test_data, y_pred_test) ** 0.5
        train_r2 = r2_score(y_train_data, y_pred_train)
        test_r2 = r2_score(y_test_data, y_pred_test)
        
        mean_val = np.mean(y_test_data)
        rel_rmse = (test_rmse / mean_val) * 100 if mean_val != 0 else 0
        
        pbar.update(1)
        pbar.set_description(f"✨ {name} Complete")

    all_results.append({
        "Model": name,
        "RMSE Test": round(test_rmse, 2),
        "R² Test": round(test_r2, 3),
        "Rel. RMSE %": f"{round(rel_rmse, 1)}%",
        "RMSE Train": round(train_rmse, 2),
        "R² Train": round(train_r2, 3)
    })

    return y_pred_test

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

# --- 1. EXECUTING MODELS (Morgan Features) ---
preds_rf = train_test_model(ranf_reg, X_train_morgan, y_train, X_test_morgan, y_test, name="Random Forest (Morgan)")
preds_xgb = train_test_model(xgb_reg, X_train_morgan, y_train, X_test_morgan, y_test, name="XGBoost Base (Morgan)")
preds_xgb_GridSearchCV = train_test_model(xgb_reg_GridSearchCV, X_train_morgan, y_train, X_test_morgan, y_test, name="XGBoost GridSearchCV (Morgan)")
preds_xgb_OPTUNA = train_test_model(xgb_reg_OPTUNA, X_train_morgan, y_train, X_test_morgan, y_test, name="XGBoost OPTUNA (Morgan)")  

# --- 2. EXECUTING MACCS MODEL ---
preds_maccs = train_test_model(xgb_maccs, X_train_maccs, y_train, X_test_maccs, y_test, name="XGBoost MACCS")

# --- 3. DISPLAYING RESULTS ---
df_final_results = pd.DataFrame(all_results)
print("\n📊 COMPREHENSIVE PERFORMANCE SUMMARY:")
display(df_final_results.sort_values(by='RMSE Test'))

# --- 4. COMPARATIVE PLOTS ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
df_plot = df_final_results.set_index('Model')

# Error Plot (RMSE)
df_plot[['RMSE Train', 'RMSE Test']].plot(kind='bar', ax=ax1, color=['#7f8c8d', '#e74c3c'])
ax1.set_ylabel("RMSE")
ax1.set_title("Error Comparison (RMSE)")
ax1.tick_params(axis='x', rotation=45)
ax1.grid(axis='y', linestyle='--', alpha=0.7)

# Performance Plot (R²)
df_plot[['R² Train', 'R² Test']].plot(kind='bar', ax=ax2, color=['#95a5a6', '#2ecc71'])
ax2.set_ylabel("R² Score")
ax2.set_ylim(0, 1.1)
ax2.set_title("Predictive Capacity (R²)")
ax2.tick_params(axis='x', rotation=45)
ax2.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()